# SMS Spam Detection

This notebook develops and evaluates NLP models for classifying SMS messages as ham or spam.

## 1. Dataset Loading and Validation

The SMS Spam Collection is loaded from the UCI Machine Learning Repository file. Initial checks examine its size, labels, missing values, empty messages, and duplicates before preprocessing.

In [1]:
from pathlib import Path
import csv
import pandas as pd

pd.set_option("display.max_colwidth", 120)

data_path = Path("data/SMSSpamCollection")
if not data_path.exists():
    data_path = Path("../data/SMSSpamCollection")

if not data_path.exists():
    raise FileNotFoundError("SMSSpamCollection was not found in the data folder.")

data_path

PosixPath('data/SMSSpamCollection')

In [2]:
df = pd.read_csv(
    data_path,
    sep="\t",
    header=None,
    names=["label", "message"],
    encoding="utf-8",
    quoting=csv.QUOTE_NONE,
)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
df.head()

Rows: 5,574
Columns: 2


,label,message
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std t...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives around here though"


In [3]:
validation = pd.Series({
    "missing_labels": int(df["label"].isna().sum()),
    "missing_messages": int(df["message"].isna().sum()),
    "empty_messages": int(df["message"].fillna("").str.strip().eq("").sum()),
    "duplicate_rows": int(df.duplicated().sum()),
    "invalid_labels": int((~df["label"].isin(["ham", "spam"])).sum()),
}, name="count")

validation.to_frame()

,count
missing_labels,0
missing_messages,0
empty_messages,0
duplicate_rows,403
invalid_labels,0


In [4]:
class_counts = df["label"].value_counts().rename("count").to_frame()
class_counts["percentage"] = (class_counts["count"] / len(df) * 100).round(2)
class_counts

,count,percentage
label,,
ham,4827,86.6
spam,747,13.4


### Initial observations

The validation results are retained before removing duplicates so the report can state exactly how the original dataset was handled. Class imbalance is expected, so later evaluation will include spam precision, recall, and F1-score rather than relying on accuracy alone.